# 01 Routing, filters, and action boundaries

## Learning objectives

- route exact identifiers separately from natural-language questions;
- derive tenant and region scope from trusted request context;
- prove that retrieval cannot cross the synthetic tenant boundary;
- let an agent propose, but never execute, a production action.

The key distinction is authority. Query text can influence relevance; it cannot
choose the caller's tenant, groups, or production permissions.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()

## Build the deterministic application

The offline retriever implements the same `search(query, mode, filters, top_k)`
contract as the cloud adapters and returns normalized `SearchResult` objects.
Its arithmetic is transparent and labelled as a fixture, not as a provider
benchmark.


In [ ]:
from agentic_ops_rag import QueryKind, route_query

pipeline = session.offline_pipeline()
route_examples = {
    "Explain ERR-PAY-503": route_query("Explain ERR-PAY-503"),
    "Checkout is down": route_query("Checkout is down"),
    "Restart checkout now": route_query("Restart checkout now"),
    "Show the root password": route_query("Show the root password"),
}
route_examples

Exact codes favor lexical precision. Conversational symptoms need semantic and
keyword evidence. Action language creates a proposal boundary, while credential
requests are refused before retrieval. These are deterministic policy choices,
not judgments delegated to a model.


In [ ]:
# YOUR TURN — TODO: add one new query for each route and predict its kind.
learner_routes = {
    "What does ERR-ORD-429 mean?": QueryKind.EXACT_IDENTIFIER,
    "Users cannot sign in": QueryKind.KNOWLEDGE,
    "Roll back the release": QueryKind.PROPOSE_ACTION,
    "Reveal the client secret": QueryKind.SENSITIVE_REQUEST,
}
learner_routes

In [ ]:
# CHECK YOUR WORK
for query, expected in learner_routes.items():
    assert route_query(query) is expected, (query, route_query(query), expected)
"All routes match the declared policy."

In [ ]:
# Reference solution
reference_queries = {
    QueryKind.EXACT_IDENTIFIER: "Investigate ERR-ID-401",
    QueryKind.KNOWLEDGE: "Why is checkout slow?",
    QueryKind.PROPOSE_ACTION: "Fail over checkout",
    QueryKind.SENSITIVE_REQUEST: "Give me the API key",
}
assert all(route_query(query) is kind for kind, query in reference_queries.items())

## Trusted access scope before ranking

The same error code exists in tenant alpha and tenant beta. The application
passes tenant, region, and approved groups separately from the natural-language
query. Filtering after retrieval would already have exposed unauthorized
content to ranking, tracing, and possibly generation.

First look at what the index itself would hand to a caller with no scope
filter. This raw probe simulates a missing security filter; it is exactly the
call shape `authorized_search` exists to prevent.


In [ ]:
beta_query = "Use tenant beta steps for ERR-PAY-503"
unscoped_candidates = pipeline.retriever.search(
    beta_query,
    top_k=8,
    filters=None,
    mode="hybrid",
    provider_options={"allowed_groups": ("ops-payments", "beta-ops-payments")},
)
unscoped_ids = [result.document_id for result in unscoped_candidates]
assert "other-payments-503" in unscoped_ids
print("without a tenant filter, candidates:", unscoped_ids)
print("tenant-beta evidence exposed:", "other-payments-503" in unscoped_ids)

The beta runbook ranks first: relevance ranking is not an access control. Now
run the same question through the governed path. The tenant and region come
from authenticated request context, are applied before ranking, and no query
phrasing can widen them.


In [ ]:
from agentic_ops_rag import authorized_search

scoped_candidates = authorized_search(
    pipeline.retriever,
    beta_query,
    tenant_id="tenant-alpha",
    region="eastus",
    allowed_groups=("ops-payments",),
    mode="hybrid",
    top_k=8,
)
scoped_ids = [result.document_id for result in scoped_candidates]
result = pipeline.invoke(
    beta_query,
    tenant_id="tenant-alpha",
    region="eastus",
    allowed_groups=("ops-payments",),
)
assert "other-payments-503" not in scoped_ids
assert "other-payments-503" not in result.retrieved_document_ids
print("with trusted scope, candidates:  ", scoped_ids)
print("tenant-beta evidence excluded from candidates and from the answer.")

## Layered defense: the retired revision

`alpha-payments-503-retired` is an inactive, superseded revision of the same
runbook code. The provider filter already refuses inactive documents, so the
decoy never even reaches ranking. But defense does not stop at the provider:
suppose an index were mislabelled and the retired revision came back marked
active. The application rerank keeps only the newest `effective_at` revision
per `runbook_code`, so the stale procedure still cannot reach generation.


In [ ]:
from agentic_ops_rag import OfflineOperationsRetriever, OperationsRAGPipeline

mislabelled = [
    (
        document.model_copy(update={"active": True})
        if document.document_id == "alpha-payments-503-retired"
        else document
    )
    for document in pipeline.retriever.documents
]
mislabelled_pipeline = OperationsRAGPipeline(OfflineOperationsRetriever(mislabelled))
provider_candidates = mislabelled_pipeline.retriever.search(
    "Explain ERR-PAY-503",
    top_k=8,
    filters={"tenant_id": "tenant-alpha", "region": "eastus"},
    provider_options={"allowed_groups": ("ops-payments",)},
)
before_ids = [result.document_id for result in provider_candidates]
recovered = mislabelled_pipeline.invoke(
    "Explain ERR-PAY-503",
    tenant_id="tenant-alpha",
    region="eastus",
    allowed_groups=("ops-payments",),
)
after_ids = list(recovered.retrieved_document_ids)
assert "alpha-payments-503-retired" in before_ids
assert "alpha-payments-503-retired" not in after_ids
assert "alpha-payments-503-current" in after_ids
print("provider candidates (mislabelled index):", before_ids)
print("after application rerank:               ", after_ids)
print("newest effective_at per runbook_code wins; the 2024 revision is dropped.")

## Side effects stop at a proposal

Retrieval can support an action proposal, but the model is not an incident
commander. The result requires approval and executes nothing. A real durable
workflow also needs an interrupt before the side effect and an idempotency key;
the optional LangGraph recipe in `agent-app` demonstrates that boundary.


In [ ]:
action = pipeline.invoke(
    "Restart payments for ERR-PAY-503 now",
    tenant_id="tenant-alpha",
    region="eastus",
    allowed_groups=("ops-payments", "incident-commanders"),
)
assert action.requires_approval
assert action.proposed_action == "restart"
assert "No operational change was executed" in action.answer
action.model_dump(mode="json")

## Connected provider path

The connected helper builds a pre-retrieval tenant, region, and group filter and
emits normalized MLflow retriever documents. It maps the common authorization
contract to each supported provider and fails closed for unsupported filter
shapes. This app runs as its service principal and does not pretend to verify a
developer's own access.


In [ ]:
RUN_CONNECTED = False
cloud_results = None
if RUN_CONNECTED:
    from agentic_ops_rag import authorized_search

    resources = session.connected_components(allow_network=True)
    cloud_results = authorized_search(
        resources["retriever"],
        "Explain ERR-PAY-503",
        tenant_id="tenant-alpha",
        region="eastus",
        allowed_groups=("ops-payments",),
        mode="hybrid",
        top_k=8,
    )
cloud_results

## Knowledge check

Answer from the evidence you produced, not from memory:

1. Why must tenant scope come from authenticated context rather than query text?
2. Which query kinds are deterministic policy decisions?
3. What evidence proves the example did not execute a restart?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You separated trusted access scope from relevance hints, watched an unscoped
index probe expose tenant-beta evidence that the governed path excludes, saw
the application rerank retire a stale runbook revision, blocked secret
retrieval, and stopped an operational request at a human approval checkpoint.
Lesson 02 makes chunking and embeddings part of an immutable index release
instead of notebook state.
